# QuetHoaDon · Tuần 5: demo đầu-cuối, độ trễ (RQ3), tiền xử lý OCR (RQ1)

Chạy trên Kaggle (GPU T4, Internet bật). Gắn dataset `quethoadon` (ảnh đã tiền xử lý + token dataset) và `m1b-v2-result` (checkpoint M1b).
Phần RQ1 cần thêm dataset công khai **Vietnamese Receipts MC OCR 2021** (ảnh gốc): Add Input → tìm `vietnamese-receipts-mc-ocr-2021`.
Mỗi cell bắt đầu bằng `# CELL n`; cell kiểm thử ghi rõ kết quả mong đợi. Chạy lần lượt, dừng ở cell lỗi và gửi output.

In [ ]:
# CELL 1 - Cài detectron2 (LayoutXLM cần). Mong đợi: "detectron2 OK" (lần đầu khoảng 5-10 phút)
try:
    import detectron2
    print("detectron2 đã có:", detectron2.__version__)
except ImportError:
    !pip install -q 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation 2>&1 | tail -3
    import detectron2; print("detectron2 OK", detectron2.__version__)

In [ ]:
# CELL 2 - Lấy mã nguồn từ nhánh main. Mong đợi: một dòng commit của main
BRANCH = "main"
import os
%cd /kaggle/working
if not os.path.exists('QuetHoaDon'):
    !git clone -q https://github.com/CaoLePhucThinh2203/QuetHoaDon.git
%cd /kaggle/working/QuetHoaDon
!git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH} && git log --oneline -1
!pip install -q -r requirements-dev.txt

In [ ]:
# CELL 3 [KIỂM THỬ] - Tìm dữ liệu, kiểm tra đúng checkpoint đã chấm ở Tuần 4, chạy unit test.
# Mong đợi: số ảnh 1153; CKPT nằm trong runs/m1b_v2_20261110_183118; "Ran ... tests" rồi "OK"
import glob, os
TOK = os.path.dirname(glob.glob('/kaggle/input/**/token_dataset_v1*/**/validation.jsonl', recursive=True)[0])
IMG = os.path.dirname(glob.glob('/kaggle/input/**/preprocessed_images*/**/*.jpg', recursive=True)[0])
found = sorted(glob.glob('/kaggle/input/**/runs/m1b_v2_*/best', recursive=True))
CKPT = found[-1] if found else None
EVAL_RUN = "m1b_v2_20261110_183118"          # run whose scores are in the Week-4 report
print("TOK =", TOK, "->", sorted(os.listdir(TOK)))
print("IMG =", IMG, "| số ảnh:", len(os.listdir(IMG)))
print("CKPT =", CKPT, "| các run tìm thấy:", [os.path.basename(os.path.dirname(f)) for f in found])
assert CKPT and os.path.basename(os.path.dirname(CKPT)) == EVAL_RUN, "checkpoint không phải run đã chấm ở Tuần 4"
os.makedirs('/kaggle/working/eval', exist_ok=True)
!python -m unittest discover -s tests 2>&1 | grep -E "^(Ran|OK|FAILED|ERROR|FAIL)"
# Nếu có test lỗi: chạy lại với "2>&1 | tail -40" và gửi phần lỗi.

## 1. Demo Data → Model → Output

In [ ]:
# CELL 4 [KIỂM THỬ] - Demo không cần OCR: lấy từ và toạ độ đã có của 5 hóa đơn validation, chạy M1b, so với nhãn chuẩn.
# Mong đợi: 5 khối, mỗi khối có 4 trường dự đoán (text, status, source) và dòng "nhãn chuẩn" để đối chiếu bằng mắt.
import sys; sys.path.insert(0, '/kaggle/working/QuetHoaDon')
from src.pipeline.extractor import ReceiptExtractor
from src.training.data import load_jsonl
extractor = ReceiptExtractor.from_checkpoint(CKPT, '/kaggle/working/QuetHoaDon/configs', with_ocr=False)
records = load_jsonl(os.path.join(TOK, 'validation.jsonl'))
for rec in records[:5]:
    out = extractor.extract_from_record(rec, IMG)
    print("=" * 70); print(rec["img_id"], "| số từ:", out["n_words"], "| layoutxlm:", out["timing_sec"]["layoutxlm"], "s")
    for name, f in out["fields"].items():
        print(f"  {name:<11} [{f['status']}/{f['source']}] {f['text']}")
    print("  nhãn chuẩn:", {k: v for k, v in rec["gt"].items()})

### Kiểm chứng trước khi demo
Chạy lại M1b qua đúng đường code của demo trên 115 hóa đơn validation rồi so với dự đoán và điểm đã lưu ở Tuần 4.
Phải ra **PASS** thì demo mới đúng là model đã chấm điểm. Cell này không đọc tập test.

In [ ]:
# CELL 4b [KIỂM THỬ] - Demo có cho lại đúng dự đoán Tuần 4 không? (khoảng 1-3 phút)
# Mong đợi: "reference macro" = 0.6592 (nếu khác: file tham chiếu trong dataset là bản cũ, gửi mình),
#           "identical 115", "field mismatches" toàn 0, dòng cuối PASS
import json
REF = glob.glob('/kaggle/input/**/eval/m1b_v2_val.json', recursive=True)[0]
ref = json.load(open(REF))
print("REF =", REF, "| reference macro =", round(ref["bootstrap_soft_rules"]["macro_f1"]["value"], 4))
!python scripts/verify_extractor.py --checkpoint "{CKPT}" --token-dir "{TOK}" --image-dir "{IMG}" --reference "{REF}" --out /kaggle/working/eval/verify_extractor_val.json

In [ ]:
# CELL 5 - Cài thư viện OCR (CRAFT, VietOCR). Mong đợi: không lỗi (VietOCR tải trọng số lần đầu dùng)
!pip install -q craft-text-detector vietocr --no-deps
!pip install -q gdown einops prefetch_generator albumentations lmdb imgaug 2>&1 | tail -2
print("xong")

In [ ]:
# CELL 6 [KIỂM THỬ] - Demo đầu-cuối từ ẢNH: tiền xử lý → CRAFT → VietOCR → LayoutXLM → 4 trường.
# Mong đợi: JSON có "fields" và "timing_sec" (preprocess, detect, ocr, layoutxlm, total). Dùng ảnh mẫu của repo.
import json
extractor = ReceiptExtractor.from_checkpoint(CKPT, '/kaggle/working/QuetHoaDon/configs')
result = extractor.extract('/kaggle/working/QuetHoaDon/assets/samples/sample_receipt.jpg')
result.pop("words")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# CELL 7 - Demo từ ảnh trên 3 ảnh GỐC (cần gắn dataset MC-OCR); nếu chưa gắn thì dùng ảnh đã tiền xử lý và ghi rõ
raw = sorted(glob.glob('/kaggle/input/**/train_images/**/*.jpg', recursive=True))
paths = raw[:3] if raw else sorted(glob.glob(IMG + '/*.jpg'))[:3]
print("ảnh gốc MC-OCR" if raw else "CHÚ Ý: ảnh đã tiền xử lý, pipeline sẽ tiền xử lý lần nữa (không giống lúc huấn luyện)")
for p in paths:
    r = extractor.extract(p); print(os.path.basename(p), {k: v["text"] for k, v in r["fields"].items()}, r["timing_sec"])

## 2. Độ trễ (RQ3)

In [ ]:
# CELL 8 - Đo độ trễ giai đoạn LayoutXLM của M1b (và M1a, B0 nếu có) trên GPU và CPU. Mất khoảng 5-15 phút (CPU chậm).
# Thêm --checkpoint M1a=<đường dẫn> và --b0-model <file .pkl> nếu đã có; mặc định chỉ đo M1b.
!python scripts/measure_latency.py --token-dir "{TOK}" --image-dir "{IMG}" --checkpoint "M1b={CKPT}" --out /kaggle/working/eval/latency.json

## 3. Tiền xử lý và nhận dạng chữ (RQ1)
Đo trên **tập validation** (không đọc test). Mỗi vùng có nhãn được cắt theo dòng nhãn rồi cho VietOCR đọc, nên kết quả nói về
khâu nhận dạng dưới từng cấu hình tiền xử lý, không phải lỗi OCR toàn pipeline (pipeline thật đọc theo box của CRAFT).
Ghi đúng kết luận theo bảng, kể cả khi tiền xử lý không giúp.

In [ ]:
# CELL 9 [KIỂM THỬ] - Tìm ảnh gốc MC-OCR, tạo split và kiểm tra split trùng với split đã dùng để train.
# Mong đợi: số ảnh gốc in ra; "validation khớp token dataset: True (115 hóa đơn)". Nếu False thì DỪNG và gửi output.
import csv
from src.training.data import load_jsonl
CSV = glob.glob('/kaggle/input/**/mcocr_train_df.csv', recursive=True)[0]
IMG_RAW = os.path.dirname(glob.glob('/kaggle/input/**/train_images/**/*.jpg', recursive=True)[0])
print("CSV =", CSV); print("IMG_RAW =", IMG_RAW, "| số ảnh:", len(os.listdir(IMG_RAW)))
!python scripts/prepare_dataset.py --csv "{CSV}" --images "{IMG_RAW}" --out /kaggle/working/dataset > /kaggle/working/dataset_summary.txt
with open('/kaggle/working/dataset/mcocr_manifest_split.csv', encoding='utf-8') as f:
    new_val = {r['img_id'] for r in csv.DictReader(f) if r['split'] == 'validation'}
tok_val = {r['img_id'] for r in load_jsonl(os.path.join(TOK, 'validation.jsonl'))}
print("validation khớp token dataset:", new_val == tok_val, f"({len(new_val)} hóa đơn; token dataset {len(tok_val)})")

In [ ]:
# CELL 10 [KIỂM THỬ] - Chạy thử 10 hóa đơn (1-2 phút). Mong đợi: bảng CER/WER cho 4 cấu hình và "skipped" gần 0. Nếu lỗi, gửi output.
!python scripts/ocr_preprocessing_ablation.py --csv "{CSV}" --image-dir "{IMG_RAW}" --split-manifest /kaggle/working/dataset/mcocr_manifest_split.csv --max-receipts 10 --out /kaggle/working/eval/ocr_ablation_smoke.json

In [ ]:
# CELL 11 - Chạy đầy đủ trên 115 hóa đơn validation, thêm ảnh làm mờ / tối / nhiễu (khoảng 10-25 phút). Mong đợi: bảng CER/WER và bảng so sánh cặp.
!python scripts/ocr_preprocessing_ablation.py --csv "{CSV}" --image-dir "{IMG_RAW}" --split-manifest /kaggle/working/dataset/mcocr_manifest_split.csv --degradations none,blur,dark,noise --out /kaggle/working/eval/ocr_ablation.json --md /kaggle/working/eval/ocr_ablation.md

## 4. Ảnh gần trùng (lưu bằng chứng)

In [ ]:
# CELL 12 - Chạy lại bước tìm ảnh gần trùng và lưu kết quả. Cần các tệp chấm test của Tuần 4 trong dataset m1b-v2-result/eval.
EV = os.path.dirname(glob.glob('/kaggle/input/**/eval/m1b_v2_test.json', recursive=True)[0])
print("EV =", EV, os.listdir(EV))
!python scripts/find_duplicates.py --token-dir "{TOK}" --image-dir "{IMG}" --out /kaggle/working/eval/dup_report.json | head -14
!python scripts/rescore_without_duplicates.py --dups /kaggle/working/eval/dup_report.json --report B0={EV}/b0_test.json --report M1a={EV}/m1a_test.json --report M1b={EV}/m1b_v2_test.json --out /kaggle/working/eval/rescore_without_duplicates.md

## 5. Lưu kết quả

In [ ]:
# CELL 13 - Nén kết quả để tải về (tab Output → week5_eval.zip)
!cd /kaggle/working && zip -q -r week5_eval.zip eval && ls -la week5_eval.zip